# Visualizacion y creacion de tabla

In [ ]:
def agregar_variables(data: pd.DataFrame):
    """
    Agrega nuevas variables al set de datos original mediante ingeniería de características.

    Creamos la variable 'ratio_deuda' (deuda_total / ingreso_mensual) y
    una clasificación ordinal 'riesgo_financiero' ('Alto' o 'Bajo' según si la deuda supera el 50% del ingreso).

    Justificación de negocio: Estas variables aportan gran valor predictivo para el modelo.
    Un cliente con una alta carga financiera (ratio > 0.5) está asfixiado económicamente y
    es sumamente propenso a cancelar suscripciones de servicios digitales para ahorrar,
    impactando directamente en la tasa de abandono que la empresa busca reducir.

    Parámetros:
        data (pd.DataFrame): El conjunto de datos base que contiene la información de los clientes.

    Retorna:
        pd.DataFrame: Un nuevo DataFrame con las columnas calculadas añadidas.
    """
    # Trabajaremos sobre una copia para no alterar el DataFrame original accidentalmente
    data = data.copy()

    # Primero calculamos la proporción de la deuda frente al ingreso (Variable Numérica)
    data['ratio_deuda'] = data['deuda_total'] / data['ingreso_mensual']

    # Luego clasificamos el riesgo financiero usando numpy (Variable Categórica Ordinal)
    # Si ratio_deuda es mayor a 0.5, asigna 'Alto', de lo contrario asigna 'Bajo'
    data['riesgo_financiero'] = np.where(data['ratio_deuda'] > 0.5, 'Alto', 'Bajo')

    return data

df = agregar_variables(df)
df

,id_cliente,fecha_registro,edad,genero,region,estado_civil,ingreso_mensual,gasto_mensual,deuda_total,score_crediticio,...,tipo_plan,num_productos,tiene_tarjeta_credito,canal_registro,dia_semana_registro,hora_registro,codigo_postal,abandono,ratio_deuda,riesgo_financiero
0,1,2021-10-27,66,Otro,Norte,Divorciado,9.243057e+05,524088.303055,2.448145e+06,455.406680,...,Estandar,3,1,Tienda,Lunes,22,3824,1,2.648631,Alto
1,2,2018-08-25,51,Masculino,Centro,Soltero,1.384687e+06,314259.751474,1.620569e+06,575.048508,...,Premium,4,1,App,Martes,10,4148,0,1.170350,Alto
2,3,2019-05-25,48,Femenino,Norte,Casado,NaN,387192.316142,5.395040e+06,770.716904,...,Premium,4,1,App,Jueves,6,7200,0,NaN,Bajo
3,4,2022-04-20,54,Masculino,Sur,Casado,4.369032e+05,417328.601856,2.999350e+06,442.722671,...,Estandar,2,1,App,Domingo,16,1782,1,6.865021,Alto
4,5,2020-03-19,31,Otro,Centro,Soltero,7.408561e+05,490961.191253,1.637711e+06,468.188403,...,Estandar,3,1,Web,Martes,8,3448,1,2.210565,Alto
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20395,4925,2021-08-09,47,Femenino,Centro,Divorciado,4.818457e+05,523321.283921,1.750852e+06,715.276140,...,Basico,1,0,Web,Viernes,4,5831,0,3.633635,Alto
20396,1627,2021-10-26,18,Masculino,Sur,Divorciado,1.062000e+06,692112.449676,1.865442e+06,662.855074,...,Premium,3,0,Web,Domingo,17,2925,0,1.756537,Alto
20397,13007,2022-10-05,76,Femenino,Norte,Soltero,1.595617e+06,568544.699045,1.862204e+06,640.595219,...,Estandar,4,0,App,Viernes,16,1660,0,1.167075,Alto
20398,17724,2020-01-10,23,Otro,Sur,Casado,8.056724e+05,348930.779211,2.032151e+06,466.372014,...,Premium,2,0,Tienda,Lunes,14,4561,1,2.522305,Alto


7) Configuracion y aplicacion de Winsorizer y nuestro Pipeline

In [ ]:
# para lidiar con los valores atipicos usamos WINSORIZER, este codigo nos permite acortar el "piso y techo" de nuestros datos, en este caso, ajustando los datos en un 5% (al interior) respectivamente
class Winsorizer(BaseEstimator, TransformerMixin):
  """
  Tratamiento de atípicos

  Parámetros
  ----------
  BaseEstimator : Clase base para estimadores en scikit-learn.
  TransformerMixin : Clase base para transformadores en scikit-learn.

  Atributos
  ---------
  columns_ : array-like
    Nombres de las columnas a transformar.
  limits : tuple
    % de los extremos a descartar
  """
  def __init__(self, limits=(0.05, 0.05)):
    self.limits = limits

  def fit(self, X, y=None):
    # Guardar nombres si es DataFrame, si no generar nombres genéricos
    if isinstance(X, pd.DataFrame):
      self.columns_ = X.columns
    else:
      self.columns_ = np.arange(X.shape[1])
    return self

  def transform(self, X):
    X = pd.DataFrame(X, columns=self.columns_)
    for col in self.columns_:
      lower = X[col].quantile(self.limits[0])
      upper = X[col].quantile(1 - self.limits[1])
      X = X.astype("float64")
      X[col] = np.clip(X[col], lower, upper)
    return X

  def get_feature_names_out(self, input_features=None):
    if input_features is None:
      return np.array(self.columns_)
    else:
      return np.array(input_features)

In [ ]:
# funcion para eliminar duplicados, señalando el data frame, lo invocaremos en nuestro pipeline
def eliminar_duplicados(X):
  return X.drop_duplicates()

In [ ]:
numeric_features = [
    'edad',
    'ingreso_mensual',
    'gasto_mensual',
    'deuda_total',
    'score_crediticio',
    'antiguedad_meses',
    'frecuencia_compra',
    'ultima_compra_dias',
    'num_productos',
    'ratio_deuda'
]

In [ ]:
"""
DEFINICIÓN DE VARIABLES POR NATURALEZA TÉCNICA

En este bloque categorizamos las variables del dataset para su tratamiento en el Pipeline.
Es fundamental distinguir entre los 'predictores' y la 'variable objetivo'.

JUSTIFICACIÓN DE 'TIENE_TARJETA_CREDITO' VS 'ABANDONO':
Aunque ambas variables son nominales y binarias (0/1), se gestionan de forma distinta:
1. 'tiene_tarjeta_credito': Es un PREDICTOR que el modelo utiliza para aprender patrones
   de comportamiento. Por ello, se incluye en el procesamiento para ser normalizado.

2. 'abandono': Es la VARIABLE OBJETIVO (Target) que deseamos predecir.
  No se incluye en las listas de preprocesamiento para evitar el 'Data Leakage' (fuga de datos),
   asegurando que el modelo aprenda a predecir el Churn basándose en el comportamiento
   previo del cliente y no en la respuesta misma.
"""

# Definimos las variables categóricas nominales (sin orden jerárquico)
categorical_nominal_features = [
    'genero',
    'region',
    'estado_civil',
    'canal_registro',
    'dia_semana_registro',
    'tiene_tarjeta_credito'  # Funciona como nominal (Sí/No) para el aprendizaje
]

In [ ]:
categorical_ordinal_features = [
    'uso_app',          # Alto, Medio, Bajo
    'tipo_plan',        # Básico, Estándar, Premium
    'riesgo_financiero' # Alto, Bajo (Nuestra otra variable nueva)
]

Necesitamos conocer los valores unicos de las siguientes columnas para evitar problemas durante su tratamiento en el pipeline

In [ ]:
df['uso_app'].unique()

array(['Bajo', 'Medio', 'Alto'], dtype=object)

In [ ]:
df['tipo_plan'].unique()

array(['Estandar', 'Premium', 'Basico'], dtype=object)

In [ ]:
df['riesgo_financiero'].unique()

array(['Alto', 'Bajo'], dtype=object)

In [ ]:
def crear_pipeline_procesamiento():
    """
    Construye el motor de preprocesamiento automatizado (Pipeline) para el conjunto de datos.

    La estandarización, imputación y codificación se realizan INTERNAMENTE para garantizar
    la reproducibilidad y evitar el sesgo de datos (data leakage).

    JUSTIFICACIÓN DE LOS PROCESOS ELEGIDOS:

    1. Pipeline Numérico:
       - Winsorizer (0.05, 0.95): Se eligió este límite para mitigar el impacto de valores extremos
         en variables como 'deuda_total' e 'ingreso_mensual'. En lugar de eliminar filas,
         "aplastamos" los outliers al percentil 5 y 95 para conservar la información del negocio.

       - Imputación (mean): Se utiliza la media para rellenar nulos en 'gasto_mensual' e 'ingreso_mensual'
         asumiendo una distribución relativamente normal tras el winsorizing, manteniendo el centro de la masa de datos.

       - MinMaxScaler: Se seleccionó para normalizar todas las magnitudes al rango [0, 1].
         Esto es vital para que variables con escalas grandes (deuda) no opaquen a las pequeñas (edad) en el modelo.

    2. Pipeline Nominal (OneHotEncoder):
       - Se aplica a variables sin jerarquía (genero, region). Se eligió OneHot para crear columnas binarias
         independientes, permitiendo que el modelo entienda la presencia de una categoría sin asumir que
         una región es "mayor" que otra.

    3. Pipeline Ordinal (OrdinalEncoder):
       - Se utiliza para 'uso_app', 'tipo_plan' y 'riesgo_financiero' . A diferencia de OneHot,
         aquí elegimos mantener un orden numérico (0 < 1 < 2) porque existe una jerarquía lógica clara
         (Básico < Estándar < Premium) que aporta valor predictivo.

    4. Limpieza de Filas (FunctionTransformer):
       - Se integra la eliminación de duplicados para centralizar toda la lógica de limpieza
         en un solo objeto de flujo de trabajo.

    Retorna:
        Pipeline: Flujo de trabajo de Scikit-Learn listo para .fit_transform().
    """

    # 1. Pipeline para variables numéricas (Limpieza + Imputación + Escalado)
    pipeline_numerico = Pipeline(
        steps=[
            ("winsorizer", Winsorizer(limits=(0.05, 0.05))),
            ("imputacion", SimpleImputer(strategy="mean")),
            ("escalado", MinMaxScaler())
        ]
    )

    # 2. Pipeline para variables categóricas Nominales (Sin orden -> OneHot)
    pipeline_nominal = Pipeline(
        steps=[
            ("imputacion", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(sparse_output=False, handle_unknown="ignore"))
        ]
    )

    # 3. Pipeline para variables categóricas Ordinales (Con orden -> OrdinalEncoder)
    pipeline_ordinal = Pipeline(
        steps=[
            ("imputacion", SimpleImputer(strategy="most_frequent")),
            ("encoder", OrdinalEncoder(categories=[
                ['Bajo', 'Medio', 'Alto'],          # Orden para 'uso_app'
                ['Basico', 'Estandar', 'Premium'],  # Orden para 'tipo_plan'
                ['Bajo', 'Alto']                    # Orden para 'riesgo_financiero'
            ]))
        ]
    )

    # 4. Ensamblador principal: Une los 3 mini-pipelines
    preprocesador = ColumnTransformer(
        transformers=[
            ("num", pipeline_numerico, numeric_features),
            ("cat_nom", pipeline_nominal, categorical_nominal_features),
            ("cat_ord", pipeline_ordinal, categorical_ordinal_features)
        ]
    )

    # 5. Pipeline Final
    pipeline_final = Pipeline(
        steps=[
            ("duplicados", FunctionTransformer(eliminar_duplicados)),
            ("preprocesamiento", preprocesador)
        ]
    )

    return pipeline_final

mi_pipeline = crear_pipeline_procesamiento()
# 2. Pasamos nuestro DataFrame por la máquina para obtener los datos listos
data_clean_array = mi_pipeline.fit_transform(df)

creamos el data frame nuevo

In [ ]:
"""
RECONSTRUCCIÓN, LIMPIEZA FINAL Y EXPORTACIÓN DEL DATASET

Este bloque realiza la transición final desde la matriz numérica generada por el pipeline
hacia un DataFrame de Pandas estructurado y listo para el análisis o modelamiento.

Procesos incluidos:
1. Reconstrucción: Convierte la salida del pipeline en un DataFrame, recuperando los nombres
   de las columnas mediante el método 'get_feature_names_out' del preprocesador.

2. Limpieza de nombres de columna: Elimina los prefijos técnicos (num__, cat_nom__, cat_ord__)
   para que los nombres coincidan con el diccionario de datos original.

3. Consistencia de Tipos: Realiza el casting (cambiar el tipo de dato de una variable de una forma a otra)
de las variables de la lista 'numeric_features'
a formato numérico para asegurar que Pandas interprete correctamente las magnitudes.

4. Recuperación de Variable Objetivo: Se reincorpora la columna 'abandono' al set de datos limpio.
   Este paso es fundamental para el entrenamiento posterior; se realiza aplicando la misma
   lógica de eliminación de duplicados para asegurar que las 20,000 etiquetas calcen
   perfectamente con las 20,000 filas procesadas por el pipeline .

5. Persistencia: Exporta el dataset final a un archivo CSV ('dataset_clientes_limpio.csv')o.
"""

# 3. Reconstruimos el DataFrame limpio
data_clean = pd.DataFrame(
    data_clean_array,
    columns=mi_pipeline.named_steps["preprocesamiento"].get_feature_names_out()
)

# 4. Limpieza de nombres de columnas
data_clean.columns = data_clean.columns.str.replace("num__", "")
data_clean.columns = data_clean.columns.str.replace("cat_nom__", "")
data_clean.columns = data_clean.columns.str.replace("cat_ord__", "")
data_clean[numeric_features] = data_clean[numeric_features].apply(pd.to_numeric)

# --- Recuperación del Abandono ---
# Como el pipeline redujo el dataset a 20.000 filas, debemos extraer
# las 20.000 etiquetas de abandono correspondientes.
data_clean['abandono'] = df.drop_duplicates()['abandono'].values

# 5.EXPORTACIÓN: Guardamos el archivo final para GitHub
data_clean.to_csv('dataset_clientes_limpio.csv', index=False)

# Mostramos el resultado final
data_clean.head()

,edad,ingreso_mensual,gasto_mensual,deuda_total,score_crediticio,antiguedad_meses,frecuencia_compra,ultima_compra_dias,num_productos,ratio_deuda,...,dia_semana_registro_Martes,dia_semana_registro_Miercoles,dia_semana_registro_Sabado,dia_semana_registro_Viernes,tiene_tarjeta_credito_0,tiene_tarjeta_credito_1,uso_app,tipo_plan,riesgo_financiero,abandono
0,0.818182,0.638167,0.750001,0.648654,0.069273,0.886792,0.277778,1.000000,0.50,0.350003,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,1
1,0.545455,1.000000,0.328167,0.344409,0.426879,0.547170,0.388889,0.878049,0.75,0.082658,...,1.0,0.0,0.0,0.0,0.0,1.0,1.0,2.0,1.0,0
2,0.490909,0.496368,0.474789,1.000000,1.000000,0.000000,0.611111,0.649390,0.75,0.371217,...,0.0,0.0,0.0,0.0,0.0,1.0,2.0,2.0,0.0,0
3,0.600000,0.059194,0.535374,0.851296,0.031360,0.000000,0.111111,0.445122,0.25,1.000000,...,0.0,0.0,0.0,0.0,0.0,1.0,2.0,1.0,1.0,1
4,0.181818,0.420252,0.683403,0.350711,0.107477,0.066038,0.222222,0.804878,0.50,0.270779,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,1
